In [3]:
import pandas as pd

# =========================
# 0. Đọc dữ liệu
# =========================
# Nếu read_parquet lỗi thì thêm engine="fastparquet"
orders = pd.read_parquet("dataset/interim/orders_base.parquet", engine="fastparquet")
products = pd.read_parquet("dataset/interim/products_base.parquet", engine="fastparquet")
returns = pd.read_parquet("dataset/interim/returns_base.parquet", engine="fastparquet")
traffic = pd.read_parquet("dataset/interim/web_traffic_base.parquet", engine="fastparquet")

print("orders columns:", orders.columns.tolist())
print("products columns:", products.columns.tolist())
print("returns columns:", returns.columns.tolist())
print("traffic columns:", traffic.columns.tolist())


# =========================
# Q1. Median inter-order gap
# =========================
# Giả sử:
# - cột khách hàng: customer_id
# - cột ngày đặt hàng: order_date
# Nếu khác tên thì sửa lại cho đúng

orders["order_date"] = pd.to_datetime(orders["order_date"])
orders_q1 = orders.sort_values(["customer_id", "order_date"]).copy()

orders_q1["gap_days"] = (
    orders_q1.groupby("customer_id")["order_date"]
    .diff()
    .dt.days
)

median_gap = orders_q1["gap_days"].dropna().median()
print("\nQ1 - Median inter-order gap:", median_gap)

if abs(median_gap - 30) == min(abs(median_gap - 30), abs(median_gap - 90), abs(median_gap - 180), abs(median_gap - 365)):
    print("Q1 đáp án gần nhất: A) 30 ngày")
elif abs(median_gap - 90) == min(abs(median_gap - 30), abs(median_gap - 90), abs(median_gap - 180), abs(median_gap - 365)):
    print("Q1 đáp án gần nhất: B) 90 ngày")
elif abs(median_gap - 180) == min(abs(median_gap - 30), abs(median_gap - 90), abs(median_gap - 180), abs(median_gap - 365)):
    print("Q1 đáp án gần nhất: C) 180 ngày")
else:
    print("Q1 đáp án gần nhất: D) 365 ngày")


# =========================
# Q2. Segment có gross margin trung bình cao nhất
# formula = (price - cogs) / price
# =========================
# Giả sử:
# - segment
# - price
# - cogs

products_q2 = products.copy()
products_q2 = products_q2[products_q2["price"].notna() & products_q2["cogs"].notna()]
products_q2 = products_q2[products_q2["price"] != 0]

products_q2["gross_margin"] = (products_q2["price"] - products_q2["cogs"]) / products_q2["price"]

q2_result = (
    products_q2.groupby("segment", dropna=False)["gross_margin"]
    .mean()
    .sort_values(ascending=False)
)

print("\nQ2 - Mean gross margin by segment:")
print(q2_result)
print("Q2 đáp án:", q2_result.index[0])


# =========================
# Q3. Lý do trả hàng phổ biến nhất cho Streetwear
# =========================

q3_df = returns.merge(
    products[["product_id", "category"]],
    on="product_id",
    how="inner"
)

print("Unique categories sample:")
print(q3_df["category"].dropna().unique()[:20])

streetwear_df = q3_df[q3_df["category"].astype(str).str.strip().str.lower() == "streetwear"]

q3_result = streetwear_df["return_reason"].value_counts()

print("\nQ3 - Return reason counts for Streetwear:")
print(q3_result)

if len(q3_result) > 0:
    print("Q3 đáp án:", q3_result.index[0])
else:
    print("Q3: Không có bản ghi Streetwear sau khi join. Kiểm tra lại tên cột/category.")


# =========================
# Q4. traffic_source có bounce_rate trung bình thấp nhất
# =========================
# Giả sử:
# - traffic_source
# - bounce_rate

q4_result = (
    traffic.groupby("traffic_source", dropna=False)["bounce_rate"]
    .mean()
    .sort_values(ascending=True)
)

print("\nQ4 - Mean bounce_rate by traffic_source:")
print(q4_result)
print("Q4 đáp án:", q4_result.index[0])

orders columns: ['order_id', 'order_date', 'customer_id', 'zip', 'order_status', 'payment_method', 'device_type', 'order_source']
products columns: ['product_id', 'product_name', 'category', 'segment', 'size', 'color', 'price', 'cogs']
returns columns: ['return_id', 'order_id', 'product_id', 'return_date', 'return_reason', 'return_quantity', 'refund_amount']
traffic columns: ['date', 'sessions', 'unique_visitors', 'page_views', 'bounce_rate', 'avg_session_duration_sec', 'traffic_source']

Q1 - Median inter-order gap: 144.0
Q1 đáp án gần nhất: C) 180 ngày

Q2 - Mean gross margin by segment:
segment
standard       0.313442
premium        0.285377
all-weather    0.284176
activewear     0.265600
performance    0.263650
balanced       0.258038
trendy         0.240758
everyday       0.236343
Name: gross_margin, dtype: float64
Q2 đáp án: standard
Unique categories sample:
['streetwear' 'genz' 'outdoor' 'casual']

Q3 - Return reason counts for Streetwear:
return_reason
wrong_size          7626

In [4]:
import pandas as pd

# đọc file
order_items = pd.read_parquet("dataset/interim/order_items_base.parquet", engine="fastparquet")
customers = pd.read_parquet("dataset/interim/customers_base.parquet", engine="fastparquet")
orders = pd.read_parquet("dataset/interim/orders_base.parquet", engine="fastparquet")

print("order_items columns:", order_items.columns.tolist())
print("customers columns:", customers.columns.tolist())
print("orders columns:", orders.columns.tolist())


# =========================
# Q5. % dòng order_items có promo_id không null
# =========================
promo_pct = order_items["promo_id"].notna().mean() * 100
print("\nQ5 - % order_items có promo_id:", promo_pct)

choices_q5 = {
    "A) 12%": 12,
    "B) 25%": 25,
    "C) 39%": 39,
    "D) 54%": 54
}
q5_answer = min(choices_q5.items(), key=lambda x: abs(promo_pct - x[1]))
print("Q5 đáp án gần nhất:", q5_answer[0])


# =========================
# Q6. age_group nào có số đơn trung bình / khách hàng cao nhất
# =========================
# Ý tưởng:
# - lấy khách có age_group không null
# - join customers với orders theo customer_id
# - đếm số đơn theo age_group
# - chia cho số khách hàng unique trong age_group

cust_nonnull = customers[customers["age_group"].notna()].copy()

q6_df = cust_nonnull.merge(
    orders[["customer_id", "order_id"]],
    on="customer_id",
    how="left"
)

q6_summary = (
    q6_df.groupby("age_group")
    .agg(
        total_orders=("order_id", "count"),
        total_customers=("customer_id", "nunique")
    )
)

q6_summary["avg_orders_per_customer"] = (
    q6_summary["total_orders"] / q6_summary["total_customers"]
)

q6_summary = q6_summary.sort_values("avg_orders_per_customer", ascending=False)

print("\nQ6 - Avg orders per customer by age_group:")
print(q6_summary)

if not q6_summary.empty:
    print("Q6 đáp án:", q6_summary.index[0])

    q6_map = {
        "55+": "A) 55+",
        "25–34": "B) 25–34",
        "25-34": "B) 25–34",
        "35–44": "C) 35–44",
        "35-44": "C) 35–44",
        "45–54": "D) 45–54",
        "45-54": "D) 45–54",
    }
    print("Q6 final:", q6_map.get(q6_summary.index[0], q6_summary.index[0]))
else:
    print("Q6: không có dữ liệu sau khi group.")

order_items columns: ['order_id', 'product_id', 'quantity', 'discount_amount', 'promo_id', 'promo_id_2', 'unit_price']
customers columns: ['customer_id', 'zip', 'city', 'signup_date', 'gender', 'age_group', 'acquisition_channel']
orders columns: ['order_id', 'order_date', 'customer_id', 'zip', 'order_status', 'payment_method', 'device_type', 'order_source']

Q5 - % order_items có promo_id: 38.663379290368894
Q5 đáp án gần nhất: C) 39%

Q6 - Avg orders per customer by age_group:
           total_orders  total_customers  avg_orders_per_customer
age_group                                                        
55+               72760            13457                 5.406851
45-54            124138            23172                 5.357241
35-44            170368            31920                 5.337343
25-34            190622            36342                 5.245226
18-24             89057            17039                 5.226656
Q6 đáp án: 55+
Q6 final: A) 55+


In [14]:
import pandas as pd

orders = pd.read_parquet("dataset/interim/orders_base.parquet", engine="fastparquet")
order_items = pd.read_parquet("dataset/interim/order_items_base.parquet", engine="fastparquet")
geography = pd.read_parquet("dataset/interim/geography_base.parquet", engine="fastparquet")

# Chuẩn hóa zip trước khi join
orders["zip"] = orders["zip"].astype(str).str.strip()
geography["zip"] = geography["zip"].astype(str).str.strip()

# Tính revenue cho từng dòng order item
order_items["line_revenue"] = (
    order_items["quantity"] * order_items["unit_price"] - order_items["discount_amount"]
)

# Nếu muốn an toàn hơn với null
order_items["line_revenue"] = order_items["line_revenue"].fillna(0)

# Cộng doanh thu theo order
order_revenue = (
    order_items.groupby("order_id", as_index=False)["line_revenue"]
    .sum()
    .rename(columns={"line_revenue": "order_revenue"})
)

# Gắn order với zip
q7_df = order_revenue.merge(
    orders[["order_id", "zip"]],
    on="order_id",
    how="left"
)

# Gắn zip với region
q7_df = q7_df.merge(
    geography[["zip", "region"]].drop_duplicates(),
    on="zip",
    how="left"
)

# Tổng revenue theo region
q7_result = (
    q7_df.groupby("region", dropna=False)["order_revenue"]
    .sum()
    .sort_values(ascending=False)
)

print("Q7 - Total revenue by region:")
print(q7_result)

vals = q7_result.dropna().values

if len(vals) >= 3 and (max(vals) - min(vals)) < 0.05 * max(vals):
    print("Q7 final: D) Cả ba vùng có doanh thu xấp xỉ bằng nhau")
else:
    top_region = q7_result.index[0]
    q7_map = {
        "West": "A) West",
        "Central": "B) Central",
        "East": "C) East"
    }
    print("Q7 đáp án:", top_region)
    print("Q7 final:", q7_map.get(top_region, top_region))

Q7 - Total revenue by region:
region
East       7.291151e+09
Central    4.719491e+09
West       3.670227e+09
Name: order_revenue, dtype: float64
Q7 đáp án: East
Q7 final: C) East


In [9]:
import pandas as pd

# Đọc file
geography = pd.read_parquet("dataset/interim/geography_base.parquet", engine="fastparquet")
sales = pd.read_parquet("dataset/interim/sales_base.parquet", engine="fastparquet")
orders = pd.read_parquet("dataset/interim/orders_base.parquet", engine="fastparquet")




# =========================
# Q8. Trong các đơn cancelled, payment method nào nhiều nhất?
# orders đã có payment_method sẵn
# =========================

cancelled_orders = orders[
    orders["order_status"].astype(str).str.strip().str.lower() == "cancelled"
].copy()

q8_result = (
    cancelled_orders["payment_method"]
    .astype(str).str.strip().str.lower()
    .value_counts()
)

print("\nQ8 - Payment method counts for cancelled orders:")
print(q8_result)

if not q8_result.empty:
    top_method = q8_result.index[0]
    q8_map = {
        "credit_card": "A) credit_card",
        "cod": "B) cod",
        "paypal": "C) paypal",
        "bank_transfer": "D) bank_transfer"
    }
    print("Q8 đáp án:", top_method)
    print("Q8 final:", q8_map.get(top_method, top_method))
else:
    print("Q8: Không có đơn cancelled.")


Q8 - Payment method counts for cancelled orders:
payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64
Q8 đáp án: credit_card
Q8 final: A) credit_card


In [15]:
import pandas as pd

# Đọc file
order_items = pd.read_parquet("dataset/interim/order_items_base.parquet", engine="fastparquet")
returns = pd.read_parquet("dataset/interim/returns_base.parquet", engine="fastparquet")
products = pd.read_parquet("dataset/interim/products_base.parquet", engine="fastparquet")
payments = pd.read_parquet("dataset/interim/payments_base.parquet", engine="fastparquet")

# =========================
# Q9. Size nào có return rate cao nhất
# return rate = số bản ghi returns / số dòng order_items
# join với products theo product_id để lấy size
# =========================

# mẫu số: số dòng order_items theo size
oi_size = order_items.merge(
    products[["product_id", "size"]],
    on="product_id",
    how="left"
)

denom = oi_size["size"].value_counts().sort_index()

# tử số: số bản ghi returns theo size
ret_size = returns.merge(
    products[["product_id", "size"]],
    on="product_id",
    how="left"
)

num = ret_size["size"].value_counts().sort_index()

# ghép lại và tính tỷ lệ
q9_summary = pd.DataFrame({
    "returns_count": num,
    "order_items_count": denom
}).fillna(0)

q9_summary["return_rate"] = q9_summary["returns_count"] / q9_summary["order_items_count"]

# chỉ lấy 4 size cần hỏi
q9_summary = q9_summary.loc[q9_summary.index.isin(["S", "M", "L", "XL"])]
q9_summary = q9_summary.sort_values("return_rate", ascending=False)

print("Q9 - Return rate by size:")
print(q9_summary)

if not q9_summary.empty:
    top_size = q9_summary.index[0]
    q9_map = {
        "S": "A) S",
        "M": "B) M",
        "L": "C) L",
        "XL": "D) XL"
    }
    print("Q9 đáp án:", top_size)
    print("Q9 final:", q9_map.get(top_size, top_size))
else:
    print("Q9: Không có dữ liệu size phù hợp.")


# =========================
# Q10. Installments nào có payment_value trung bình cao nhất / order
# =========================

# Nếu 1 order có nhiều dòng payment cùng installments,
# cách an toàn là cộng payment_value theo order_id + installments trước
q10_df = (
    payments.groupby(["installments", "order_id"], as_index=False)["payment_value"]
    .sum()
)

q10_result = (
    q10_df.groupby("installments")["payment_value"]
    .mean()
    .sort_values(ascending=False)
)

# chỉ lấy các lựa chọn đề bài
q10_result = q10_result[q10_result.index.isin([1, 3, 6, 12])]

print("\nQ10 - Mean payment value per order by installments:")
print(q10_result)

if not q10_result.empty:
    top_inst = q10_result.index[0]
    q10_map = {
        1: "A) 1 kỳ (trả một lần)",
        3: "B) 3 kỳ",
        6: "C) 6 kỳ",
        12: "D) 12 kỳ"
    }
    print("Q10 đáp án:", top_inst)
    print("Q10 final:", q10_map.get(top_inst, top_inst))
else:
    print("Q10: Không có dữ liệu installments phù hợp.")

Q9 - Return rate by size:
      returns_count  order_items_count  return_rate
size                                               
S              9723             172039     0.056516
L              9741             173171     0.056251
M              9820             176422     0.055662
XL            10655             193021     0.055201
Q9 đáp án: S
Q9 final: A) S

Q10 - Mean payment value per order by installments:
installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
Name: payment_value, dtype: float64
Q10 đáp án: 6
Q10 final: C) 6 kỳ


In [16]:
print("Order items size counts:")
print(denom)

print("\nReturns size counts:")
print(num)

Order items size counts:
size
L     173171
M     176422
S     172039
XL    193021
Name: count, dtype: int64

Returns size counts:
size
L      9741
M      9820
S      9723
XL    10655
Name: count, dtype: int64
